# Automated Retention Agent (GenAI)
A pipeline that uses ML to identify at-risk customers and GPT-4 to generate personalized retention emails with prompt guardrails.

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from imblearn.over_sampling import SMOTE
from openai import OpenAI

## Step 1: Load and Explore Data

In [ ]:
train_df = pd.read_csv('churn-bigml-80.csv')
test_df  = pd.read_csv('churn-bigml-20.csv')

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)
train_df.head()

In [ ]:
print(train_df['Churn'].value_counts())
sns.countplot(x='Churn', data=train_df)
plt.title('Churn Distribution (Training Set)')
plt.show()

## Step 2: Preprocess Data

In [ ]:
for df in [train_df, test_df]:
    df['International plan'] = df['International plan'].map({'Yes': 1, 'No': 0})
    df['Voice mail plan']    = df['Voice mail plan'].map({'Yes': 1, 'No': 0})

train_df = pd.get_dummies(train_df, columns=['State'], drop_first=True)
test_df  = pd.get_dummies(test_df,  columns=['State'], drop_first=True)

X_train = train_df.drop('Churn', axis=1)
y_train = train_df['Churn']
X_test  = test_df.drop('Churn', axis=1)
y_test  = test_df['Churn']

X_train, X_test = X_train.align(X_test, join='left', axis=1, fill_value=0)
print("Train:", X_train.shape, "| Test:", X_test.shape)

## Step 3: Handle Class Imbalance with SMOTE

In [ ]:
sm = SMOTE(random_state=42)
X_train_res, y_train_res = sm.fit_resample(X_train, y_train)

print("After SMOTE:")
print(pd.Series(y_train_res).value_counts())

## Step 4: Train Random Forest Classifier

In [ ]:
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train_res, y_train_res)

y_pred = rf_model.predict(X_test)
print(classification_report(y_test, y_pred))

## Step 5: Feature Importance — What Drives Churn?

In [ ]:
feat_importances = pd.Series(rf_model.feature_importances_, index=X_train.columns)
top10 = feat_importances.nlargest(10)

top10.plot(kind='barh', figsize=(10, 6))
plt.title('Top 10 Features Driving Churn')
plt.tight_layout()
plt.show()

print("\nTop churn predictor:", top10.index[0])

## Step 6: Identify High-Risk Customers

In [ ]:
y_probs = rf_model.predict_proba(X_test)[:, 1]
predictions = (y_probs >= 0.4).astype(int)  # Recall-focused threshold

high_risk_indices = [i for i, val in enumerate(predictions) if val == 1][:5]
print(f"Found {len(high_risk_indices)} high-risk customers. Generating retention emails...")

## Step 7: GPT-4 Retention Email Generator with Prompt Guardrails

In [ ]:
# Load API key from environment variable — never hardcode keys
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

def ai_retention_agent(reason_for_leaving: str, monthly_bill: float, customer_service_calls: int) -> str:
    """
    Generates a personalized retention email using GPT-4.
    Prompt guardrails prevent hallucinated pricing or unauthorized discount promises.
    """
    system_prompt = """You are a Customer Loyalty Expert writing retention emails.
GUARDRAILS (strictly follow):
- Do NOT promise specific discounts or price reductions unless explicitly told to
- Do NOT mention competitor pricing
- Do NOT make up features or services
- Keep the tone warm and professional
- Email must be exactly 2-3 sentences"""

    user_prompt = f"""A customer is at risk of leaving.
Reason: {reason_for_leaving}
Monthly bill: ${monthly_bill:.2f}
Service calls this month: {customer_service_calls}

Write a short, personalized retention email to keep them."""

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user",   "content": user_prompt}
        ],
        temperature=0.7,
        max_tokens=200
    )
    return response.choices[0].message.content

## Step 8: Generate Personalized Emails for High-Risk Customers

In [ ]:
results = []

for idx in high_risk_indices:
    customer = test_df.iloc[idx]

    calls      = int(customer['Customer service calls'])
    bill       = float(customer['Total day charge'])
    intl_plan  = bool(customer['International plan'])

    # Determine churn reason based on customer profile
    if calls > 3:
        reason = "Frequent service issues and unresolved complaints"
    elif intl_plan:
        reason = "High international calling rates"
    else:
        reason = "High billing charges"

    try:
        email = ai_retention_agent(reason, bill, calls)
        results.append({
            'Customer Index': idx,
            'Churn Risk Score': f"{y_probs[idx]:.2%}",
            'Monthly Bill': f"${bill:.2f}",
            'Service Calls': calls,
            'Identified Reason': reason,
            'AI Draft Email': email
        })

        print(f"\n{'='*60}")
        print(f"Customer Index : {idx}")
        print(f"Churn Risk     : {y_probs[idx]:.2%}")
        print(f"Bill           : ${bill:.2f} | Calls: {calls}")
        print(f"Reason         : {reason}")
        print(f"\nAI Email Draft:\n{email}")

    except Exception as e:
        print(f"Error for customer {idx}: {e}")

print(f"\n✅ Generated {len(results)} retention emails.")

## Step 9: Save Results

In [ ]:
results_df = pd.DataFrame(results)
results_df.to_csv('retention_emails.csv', index=False)
print("Results saved to retention_emails.csv")
results_df[['Customer Index', 'Churn Risk Score', 'Monthly Bill', 'Identified Reason']].head()